# Seed Detection

## Imports and Constants

In [15]:
import sys
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
import torchvision
import matplotlib.pyplot as plt

In [16]:
IMAGE_DIR = Path("seed-images").resolve() # Folder with all seed images

# Find all .jpg, .jpeg, or .png files
extensions = ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.png', '*.PNG']
image_paths = []
for extension in extensions:
    image_paths.extend(list(IMAGE_DIR.glob(extension)))
print(f"Found {len(image_paths)} images")

MIN_CONTOUR_AREA = 20   # minimum pixel area to count as a seed (might need to adjust for smaller seeds)
MAX_CONTOUR_AREA = 800  # maximum pixel area to count as a seed
DARKER_THRESHOLD = 40   # how much darker than median to count as foreground
MAX_ASPECT_RATIO = 2.2  # reject elongated detections (grass/dirt)
MIN_SOLIDITY = 0.8      # reject very irregular shapes

Found 4 images


## Preprocessing

In [17]:
def process_image(path):
    image = cv2.imread(str(path))  # Load image

    if image is None:
            print(f"Warning: Could not read image at {path}")
            return None, None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # matplotlib assumes images are RGB

    # Convert to grayscale (since seeds are similar color)
    image_grayscale = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian blur to reduce noise
    image_gaussianBlur = cv2.GaussianBlur(image_grayscale, (5,5), 0)

    # Calculate the optimal threshold value and create the thresholded image to return
    # Using Otsu's Algorithm - creates a binary mask over the image (seeds are white, background black)
    optimal_threshold_value, thresholded = cv2.threshold(image_gaussianBlur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    return image_rgb, thresholded

## Segmentation of Overlapping Seeds

In [18]:
def segment_overlapping_seeds(thresholded):

    # Dilate (inflate every white blob seed outward by 3 pixels) to make background more obvious
    kernel = np.ones((3, 3), np.uint8)
    sure_background = cv2.dilate(thresholded, kernel, iterations=3)

    # Find seed centers (for every white pixel, calculate how far it is from the nearest black pixel)
    dist_transform = cv2.distanceTransform(thresholded, cv2.DIST_L2, 5)

    # Threshold to get only the brightest 50% peaks from that heatmap (center of seeds)
    _, sure_foreground = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
    sure_foreground = np.uint8(sure_foreground)

    unknown_region = cv2.subtract(sure_background, sure_foreground)    # Region where seeds may be overlapping

    # Label markers (each sure foreground blob gets an unique ID)
    _, markers = cv2.connectedComponents(sure_foreground)

    markers += 1  # Because Watershed reserves 0 for unknown regions
    markers[unknown_region == 255] = 0 # Mark unknown as 0

    # Apply Watershed algorithm
    image_bgr = cv2.cvtColor(thresholded, cv2.COLOR_GRAY2BGR) # Transform image back to BGR

    markers = cv2.watershed(image_bgr, markers) # When 2 centers meet, Watershed draws a boundary (-1) between them

    return markers

## Debris Filtering, Seed Counting, Confidence Scoring

In [19]:
def count_and_filter(original_img, markers):
    seed_count = 0
    total_objects_found = 0
    valid_contours = []
    valid_seed_areas = []
    labels = np.unique(markers)

    for label in labels:
        # Ignore watershed boundaries and background, which will be labels 0 and 1
        if label <= 1:
            continue

        total_objects_found += 1

        # Mask the current object
        mask = np.zeros(markers.shape, dtype = "uint8")
        mask[markers == label] = 255

        # Find the current object's contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        contour = contours[0]
        area = cv2.contourArea(contour)

        # ===== Filter out the debris =====

        # Check the size of the object
        if not (MIN_CONTOUR_AREA < area < MAX_CONTOUR_AREA):
            continue

        # Check the solidity of the object (debris/dirt usually has some holes)
        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        if hull_area > 0:
            solidity = float(area) / hull_area
        else:
            solidity = 0
        if solidity < MIN_SOLIDITY:
            continue

        # Check the aspect ratio of the object
        x, y, w, h = cv2.boundingRect(contour)
        if min(w, h) > 0:
            aspect_ratio = float(max(w, h)) / float(min(w, h))
        else:
            aspect_ratio = 0
        if aspect_ratio > MAX_ASPECT_RATIO:
            continue

        # If the object passed all the above checks, it is a seed
        seed_count += 1
        valid_seed_areas.append(area)

    # ===== Calculate the confidence score =====
    # If a lot of dirt/debris was detected, confidence score is lower
    cleanliness = seed_count / total_objects_found

    # Since seeds per image are of the same species, they should be pretty uniform in size
    # lower confidence if the detected seeds are wildly varying sizes
    standard_dev = np.std(valid_seed_areas)
    mean = np.mean(valid_seed_areas)
    if mean > 0:
        variation_coefficient = standard_dev / mean
    else:
        variation_coefficient = 0

    # Invert the variation coefficient so that 1 means perfectly uniform an 0 means chaotic sizes
    uniformity = max(0, 1 - variation_coefficient)

    # Calculate the confidence score by getting the mean of the cleanliness and uniformity
    confidence = (cleanliness + uniformity) / 2

    print(f"Seed count: {seed_count}")
    print(f"Confidence score: {confidence}")

    return seed_count, confidence

## CSV Export

In [20]:
results = []

for path in image_paths:
    original_img, thresholded = process_image(path)
    if original_img is None:
        continue

    markers = segment_overlapping_seeds(thresholded)
    if markers is None:
        continue

    count, confidence = count_and_filter(original_img, markers)
    if count is None or confidence is None:
        continue

    # Append the image to the results list with the required properties
    results.append({
        "image_id": path.stem,
        "seed_count": count,
        "confidence_score": confidence
    })

# Export CSV file
results_df = pd.DataFrame(results)
results_df.to_csv("seed_count_result.csv", index=False)

if len(results_df) > 0:
    print("CSV export completed. Summary of first 10 rows:")
    print(results_df.head)
else:
    print("No image data to export.")

/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Seed count: 0
Confidence score: 0.5
Seed count: 0
Confidence score: 0.5
Seed count: 2
Confidence score: 0.4112261329652634
Seed count: 0
Confidence score: 0.5
CSV export completed. Summary of first 10 rows:
<bound method NDFrame.head of    image_id  seed_count  confidence_score
0  IMG_0037           0          0.500000
1  IMG_0033           0          0.500000
2  IMG_0032           2          0.411226
3  IMG_0031           0          0.500000>
